# Block 2 lecture — one table, one question

**The lecture's queries, as Eduardo ran them: for review after class. Nothing here is typed in class.** In the lecture
you predicted each result, and the next slide showed the query with what it printed; here you can run every cell yourself, in order, and
read the result again. The cell numbers are the ones the slides use.

Everything here runs on `data/raw/countries.csv`: the country table you built in Lab 1, one row per country per year,
1990 to 2024. (Lab 1 wrote it to `data/silver/` in the Lab 1 folder; this starter ships the same table.) The Online
Retail demonstration at the start of the lecture is not here, and the lab does not need this notebook.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # up to 400 rows in full; a longer result prints head and tail with "..." between: count it, then census it by group

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

### Cell 1 — Watch — the lecture file: one row is one country in one year

In [ ]:
con.sql("""
    SELECT COUNT(*)                        AS n_rows,
           COUNT(DISTINCT (country, year)) AS n_keys,
           COUNT(DISTINCT country)         AS countries
    FROM 'data/raw/countries.csv'
""").df()

### Cell 2 — Predict, then watch — the sentence, then the query: total CO₂ of the countries in 2024

In [ ]:
con.sql("""
    SELECT SUM(co2) AS total_co2
    FROM 'data/raw/countries.csv'
    WHERE year = 2024
""").df()

### Cell 3 — Predict, then watch — seven aggregates on one year. Predict: are `n_rows` and `n_co2` equal?

In [ ]:
con.sql("""
    SELECT COUNT(*) AS n_rows, COUNT(co2) AS n_co2, COUNT(DISTINCT country) AS n_countries,
           SUM(co2) AS total_co2, AVG(co2) AS avg_co2, MIN(co2) AS min_co2, MAX(co2) AS max_co2
    FROM 'data/raw/countries.csv'
    WHERE year = 2024
""").df()

### Cell 4 — Predict, then watch — an average with the missing values left out, and with them as zeros

In [ ]:
con.sql("""
    SELECT AVG(gdp)              AS avg_gdp,
           AVG(COALESCE(gdp, 0)) AS avg_gdp_zero
    FROM 'data/raw/countries.csv'
    WHERE year = 2022
""").df()

### Cell 5 — Watch — `GROUP BY`: total CO₂ by year

In [ ]:
con.sql("""
    SELECT year, SUM(co2) AS total_co2
    FROM 'data/raw/countries.csv'
    GROUP BY year
    ORDER BY year
""").df()

### Cell 6a — Predict, then watch — the rule `GROUP BY` enforces (this query is wrong on purpose)

In [ ]:
# The try/except prints DuckDB's refusal instead of stopping the notebook.
try:
    con.sql("""
        SELECT year, country, SUM(co2)
        FROM 'data/raw/countries.csv'
        GROUP BY year
    """).df()
except duckdb.Error as e:
    print(e)

### Cell 6b — Watch — the fix when you want a label beside a group: group by the identifier, aggregate the label

In [ ]:
# The grain stays one row per code; the name rides along inside any_value().
con.sql("""
    SELECT iso_code, any_value(country) AS country, SUM(co2) AS co2_since_1990
    FROM 'data/raw/countries.csv'
    GROUP BY iso_code
    ORDER BY co2_since_1990 DESC
    LIMIT 5
""").df()

### Cell 7 — Predict, then watch — the `NULL` group. Predict: two rows, or three?

In [ ]:
con.sql("""
    SELECT gdp > 1000000000000 AS big_economy, COUNT(*) AS countries, SUM(co2) AS co2
    FROM 'data/raw/countries.csv'
    WHERE year = 2022
    GROUP BY big_economy
    ORDER BY big_economy
""").df()

### Cell 8a — Predict, then watch — the sum identity: a breakdown adds up to its total

In [ ]:
total = con.sql("SELECT SUM(co2) FROM 'data/raw/countries.csv' WHERE year = 2022").fetchone()[0]
by_group = con.sql("""
    SELECT SUM(co2) FROM (
        SELECT gdp > 1000000000000 AS big_economy, SUM(co2) AS co2
        FROM 'data/raw/countries.csv'
        WHERE year = 2022
        GROUP BY big_economy
    )
""").fetchone()[0]
print(f"Total, 2022:                     {total:10,.3f} Mt")
print(f"The three groups, added up:      {by_group:10,.3f} Mt")

### Cell 8b — Watch — two filters that sound like they cover everything: big economies, plus the rest

In [ ]:
big = con.sql("""SELECT SUM(co2) FROM 'data/raw/countries.csv'
                 WHERE year = 2022 AND gdp > 1000000000000""").fetchone()[0]
small = con.sql("""SELECT SUM(co2) FROM 'data/raw/countries.csv'
                   WHERE year = 2022 AND gdp <= 1000000000000""").fetchone()[0]

print(f"Total, 2022:                     {total:10,.3f} Mt")
print(f"gdp > 1e12, plus gdp <= 1e12:    {big + small:10,.3f} Mt")
print(f"missing from the two filters:    {total - big - small:10,.3f} Mt")

### Cell 8c — Watch — the same total and the same three groups, compared exactly

In [ ]:
print(f"Exactly: {total!r} against {by_group!r}")
print("equal with == ?", total == by_group, "   within 0.0005 Mt ?", abs(total - by_group) < 0.0005)

### Cell 9 — Watch — a share of a total: a query inside a query

In [ ]:
con.sql("""
    SELECT country, co2,
           ROUND(100 * co2 / (SELECT SUM(co2) FROM 'data/raw/countries.csv'
                              WHERE year = 2024), 2) AS pct_of_total
    FROM 'data/raw/countries.csv'
    WHERE year = 2024
    ORDER BY co2 DESC
    LIMIT 5
""").df()

### Cell 10a — Predict, then watch — `HAVING`: the years in which fewer countries reported than in 2024. Could the condition go in `WHERE`?

In [ ]:
# The prediction: could the condition go in WHERE? The try/except prints DuckDB's answer instead of stopping.
try:
    con.sql("""
        SELECT year, COUNT(co2) AS reporting
        FROM 'data/raw/countries.csv'
        WHERE COUNT(co2) < 216
        GROUP BY year
    """).df()
except duckdb.Error as e:
    print("In WHERE, DuckDB refused:", str(e).splitlines()[0])

### Cell 10b — Watch — the same condition in `HAVING`, after the groups exist

In [ ]:
con.sql("""
    SELECT year, COUNT(co2) AS reporting
    FROM 'data/raw/countries.csv'
    GROUP BY year
    HAVING COUNT(co2) < 216
    ORDER BY year
""").df()

### Cell 11 — Predict, then watch — `AND`, `OR`, and the parentheses. The sentence: China and India, in 2024

In [ ]:
without = con.sql("""
    SELECT COUNT(*) FROM 'data/raw/countries.csv'
    WHERE year = 2024 AND country = 'China' OR country = 'India'
""").fetchone()[0]
with_parentheses = con.sql("""
    SELECT COUNT(*) FROM 'data/raw/countries.csv'
    WHERE year = 2024 AND (country = 'China' OR country = 'India')
""").fetchone()[0]
print("without parentheses:", without)
print("with parentheses:   ", with_parentheses)

### Cell 12 — Predict, then watch — every country but China, and the row that is neither

In [ ]:
print(con.sql("""SELECT COUNT(*) FROM 'data/raw/countries.csv'
                 WHERE year = 2024 AND iso_code != 'CHN'""").fetchone()[0])
print(con.sql("""SELECT COUNT(*) FROM 'data/raw/countries.csv'
                 WHERE year = 2024 AND iso_code IS DISTINCT FROM 'CHN'""").fetchone()[0])

### Cell 13a — Watch — calculated columns, aliases, and division

In [ ]:
con.sql("SELECT 7 / 2 AS slash, 7 // 2 AS double_slash, ROUND(2 / 3, 2) AS rounded").df()

### Cell 13b — Watch — a calculated column with an alias: tonnes per person

In [ ]:
con.sql("""
    SELECT country, co2, population,
           ROUND(co2 * 1000000 / population, 2) AS tonnes_per_person
    FROM 'data/raw/countries.csv'
    WHERE year = 2024
    ORDER BY co2 DESC
    LIMIT 5
""").df()

### Cell 14 — Watch — dates, the minimum: one timestamp, three ways

In [ ]:
con.sql("""
    SELECT date_trunc('month', TIMESTAMP '2010-03-17 14:05:00') AS month_start,
           year(TIMESTAMP '2010-03-17 14:05:00')                AS the_year,
           strftime(TIMESTAMP '2010-03-17 14:05:00', '%Y-%m')   AS label
""").df()

### Cell 15 — Watch — a period made from a number: decades

In [ ]:
con.sql("""
    SELECT year // 10 * 10 AS decade,
           COUNT(DISTINCT year) AS years,
           SUM(co2) AS total_co2
    FROM 'data/raw/countries.csv'
    GROUP BY decade
    ORDER BY decade
""").df()

---

## What to remember from Block 2

1. **A number is a sentence before it is a query:** a measure, over a population, after a filter. Write the sentence,
   then write the query that does all of it and nothing else.
2. **The map.** A query runs `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`, not in the order you write
   it. Each clause gets only what the one before it made. A name made with `AS` is born in `SELECT`, so standard SQL
   does not let `WHERE` use it. DuckDB does, as a shortcut (and if the name is also a column's, `WHERE` uses the
   column); this course writes the expression again.
3. **Aggregates skip `NULL`.** `COUNT(*)` counts rows; `COUNT(col)` counts values; `AVG` divides by the values that
   exist (216, not 219). `COALESCE(col, 0)` turns a missing value into a zero, and a zero is a claim.
4. **`GROUP BY`:** one row per group, and every column in `SELECT` is grouped or aggregated. A `NULL` is a group of its
   own.
5. **`WHERE` filters rows, before the groups exist; `HAVING` filters groups, after.** A condition on a count or a sum
   goes in `HAVING`.
6. **`WHERE` keeps only *true*.** `!=`, `NOT IN`, and a comparison with a `NULL` are unknown, so those rows vanish.
   `IS DISTINCT FROM` keeps them. Parenthesise every mix of `AND` and `OR`.
7. **The sum identity.** An additive measure's breakdown adds up to its total, computed by other code. Averages and
   ratios have their own: a ratio's top and bottom come from the same rows. **Compare sums with a tolerance**, never
   with `==`: two correct sums of the same numbers can differ in the tenth decimal place. Money: `abs(a - b) < 0.005`.
8. **A share of a total** is a query inside a query: `co2 / (SELECT SUM(co2) FROM … WHERE year = 2024)`. The inner
   query gives one number; the outer one divides every row by it.
9. **Build one clause at a time, and say the row count before you run it.** A number you did not predict is a question.

## Common mistakes

1. **Double quotes around text.** `WHERE country = "China"` looks for a *column* called China. Text goes in single
   quotes: `'China'`.
2. **`!=` or `NOT IN` on a column that has `NULL`s.** The rows where the column is missing vanish, with no error.
   If the sentence says *everything except*, you probably want `IS DISTINCT FROM`.
3. **A condition on a count or a sum in `WHERE`.** DuckDB refuses (*WHERE clause cannot contain aggregates*). The
   condition belongs in `HAVING`, after `GROUP BY`.
4. **`COUNT(*)` when the question counts something else.** `COUNT(*)` counts rows. If one row is not one of the things
   the question counts, say what one row is first, then use `COUNT(DISTINCT …)` of the thing.
5. **`COALESCE(col, 0)` inside an average.** It turns *unknown* into *zero* and changes the average. A zero is a claim.
6. **`AND` and `OR` without parentheses.** `AND` binds first. Put brackets around every `OR`.
7. **Checking a sum with `==`.** Compare money to the penny (`abs(a - b) < 0.005`) and tonnes to the rounding of the
   source. And a check that re-runs the query it is checking cannot fail: compute the other side another way.